In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

PROJECT_ROOT = Path.cwd().resolve().parents[1]

DATA_PATH = PROJECT_ROOT / "ml" / "data" / "eta_model_data.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print()
print(df.head())

Dataset shape: (28636, 14)

               TRIP_ID   TIMESTAMP  trip_duration_minutes  \
0  1377560292620000409  1377560292                   3.50   
1  1378733467620000184  1378733467                  13.75   
2  1397495585620000173  1397495585                  16.00   
3  1391343649620000395  1391343649                   5.50   
4  1386168857620000510  1386168857                  54.00   

   straight_line_distance_km  start_longitude  start_latitude  end_longitude  \
0                   1.242103        -8.638407       41.170815      -8.638164   
1                   9.468056        -8.649585       41.154102      -8.670303   
2                   3.856975        -8.606961       41.150142      -8.580447   
3                   2.344062        -8.654787       41.153337      -8.627346   
4                  63.895258        -8.585694       41.148891      -8.821026   

   end_latitude  hour  day_of_week  is_weekend CALL_TYPE  ORIGIN_CALL  \
0     41.159646    23            0           0     

In [2]:
# ============================================================
# Cell 2 — Prepare ETA prediction features
# ============================================================

# Sort chronologically so future trips are never used
# to predict earlier trips.
df = df.sort_values("TIMESTAMP").reset_index(drop=True)

TARGET = "trip_duration_minutes"

# Features available when an ETA prediction is requested.
#
# We deliberately DO NOT use:
# - TRIP_ID                  -> identifier only
# - TIMESTAMP                -> replaced by time-derived features
# - distance_km              -> completed route distance (leakage)
# - average_speed_kmh        -> completed-trip information
# - trajectory_points       -> completed-trip information
# - POLYLINE                 -> completed trajectory
#
# The destination is reconstructed from the terminal trajectory
# because the Porto dataset does not provide an explicit destination.
# This is documented as a dataset limitation.

NUMERIC_FEATURES = [
    "straight_line_distance_km",
    "start_longitude",
    "start_latitude",
    "end_longitude",
    "end_latitude",
    "hour",
    "day_of_week",
    "is_weekend",
]

CATEGORICAL_FEATURES = [
    "CALL_TYPE",
    "ORIGIN_STAND",
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = df[FEATURES].copy()
y = df[TARGET].copy()

# Chronological 80/20 split
split_index = int(len(df) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("=== DATA SPLIT ===")
print(f"Total samples : {len(df):,}")
print(f"Training      : {len(X_train):,}")
print(f"Testing       : {len(X_test):,}")

print("\n=== TIME RANGE ===")
print(
    "Training:",
    pd.to_datetime(df["TIMESTAMP"].iloc[0], unit="s"),
    "→",
    pd.to_datetime(df["TIMESTAMP"].iloc[split_index - 1], unit="s"),
)

print(
    "Testing :",
    pd.to_datetime(df["TIMESTAMP"].iloc[split_index], unit="s"),
    "→",
    pd.to_datetime(df["TIMESTAMP"].iloc[-1], unit="s"),
)

print("\n=== FEATURES ===")
print(FEATURES)

print("\n=== TARGET ===")
print(y_train.describe().round(2))

=== DATA SPLIT ===
Total samples : 28,636
Training      : 22,908
Testing       : 5,728

=== TIME RANGE ===
Training: 2013-07-01 00:40:41 → 2014-04-25 05:01:22
Testing : 2014-04-25 05:04:42 → 2014-06-30 23:39:19

=== FEATURES ===
['straight_line_distance_km', 'start_longitude', 'start_latitude', 'end_longitude', 'end_latitude', 'hour', 'day_of_week', 'is_weekend', 'CALL_TYPE', 'ORIGIN_STAND']

=== TARGET ===
count    22908.00
mean        12.05
std          9.21
min          1.00
25%          7.00
50%         10.25
75%         14.50
max        179.00
Name: trip_duration_minutes, dtype: float64


In [3]:
# ============================================================
# Cell 3 — Preprocessing pipeline
# ============================================================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, NUMERIC_FEATURES),
        ("categorical", categorical_transformer, CATEGORICAL_FEATURES),
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [4]:
# ============================================================
# Cell 4 — Dummy baseline
# ============================================================

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            DummyRegressor(strategy="mean"),
        ),
    ]
)

baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions,
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_predictions,
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_predictions,
)

print("=== BASELINE RESULTS ===")
print(f"MAE  : {baseline_mae:.3f} minutes")
print(f"RMSE : {baseline_rmse:.3f} minutes")
print(f"R²   : {baseline_r2:.3f}")

=== BASELINE RESULTS ===
MAE  : 5.212 minutes
RMSE : 8.513 minutes
R²   : -0.000


In [5]:
# ============================================================
# Cell 5 — Linear Regression
# ============================================================

linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)

linear_mae = mean_absolute_error(
    y_test,
    linear_predictions,
)

linear_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        linear_predictions,
    )
)

linear_r2 = r2_score(
    y_test,
    linear_predictions,
)

print("=== LINEAR REGRESSION RESULTS ===")
print(f"MAE  : {linear_mae:.3f} minutes")
print(f"RMSE : {linear_rmse:.3f} minutes")
print(f"R²   : {linear_r2:.3f}")

print("\n=== IMPROVEMENT OVER BASELINE ===")
print(
    f"MAE improvement: "
    f"{baseline_mae - linear_mae:.3f} minutes"
)

=== LINEAR REGRESSION RESULTS ===
MAE  : 4.140 minutes
RMSE : 7.525 minutes
R²   : 0.219

=== IMPROVEMENT OVER BASELINE ===
MAE improvement: 1.072 minutes


In [6]:
# ============================================================
# Cell 6 — Gradient Boosting Regressor
# ============================================================

gradient_boosting_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            GradientBoostingRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=3,
                random_state=42,
                loss="huber",
            ),
        ),
    ]
)

gradient_boosting_model.fit(X_train, y_train)

gb_predictions = gradient_boosting_model.predict(X_test)

gb_mae = mean_absolute_error(
    y_test,
    gb_predictions,
)

gb_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        gb_predictions,
    )
)

gb_r2 = r2_score(
    y_test,
    gb_predictions,
)

print("=== GRADIENT BOOSTING RESULTS ===")
print(f"MAE  : {gb_mae:.3f} minutes")
print(f"RMSE : {gb_rmse:.3f} minutes")
print(f"R²   : {gb_r2:.3f}")

print("\n=== IMPROVEMENT OVER BASELINE ===")
print(
    f"MAE improvement: "
    f"{baseline_mae - gb_mae:.3f} minutes"
)

=== GRADIENT BOOSTING RESULTS ===
MAE  : 3.544 minutes
RMSE : 7.324 minutes
R²   : 0.260

=== IMPROVEMENT OVER BASELINE ===
MAE improvement: 1.668 minutes


In [7]:
# ============================================================
# Cell 7 — Random Forest Regressor
# ============================================================

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                max_depth=15,
                min_samples_leaf=3,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

random_forest_model.fit(X_train, y_train)

rf_predictions = random_forest_model.predict(X_test)

rf_mae = mean_absolute_error(
    y_test,
    rf_predictions,
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_predictions,
    )
)

rf_r2 = r2_score(
    y_test,
    rf_predictions,
)

print("=== RANDOM FOREST RESULTS ===")
print(f"MAE  : {rf_mae:.3f} minutes")
print(f"RMSE : {rf_rmse:.3f} minutes")
print(f"R²   : {rf_r2:.3f}")

print("\n=== IMPROVEMENT OVER BASELINE ===")
print(
    f"MAE improvement: "
    f"{baseline_mae - rf_mae:.3f} minutes"
)

=== RANDOM FOREST RESULTS ===
MAE  : 3.778 minutes
RMSE : 7.241 minutes
R²   : 0.277

=== IMPROVEMENT OVER BASELINE ===
MAE improvement: 1.434 minutes


In [8]:
# ============================================================
# Cell 8 — Model Comparison
# ============================================================

model_comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Linear Regression",
        "Gradient Boosting",
        "Random Forest",
    ],
    "MAE_minutes": [
        baseline_mae,
        linear_mae,
        gb_mae,
        rf_mae,
    ],
    "RMSE_minutes": [
        baseline_rmse,
        linear_rmse,
        gb_rmse,
        rf_rmse,
    ],
    "R2": [
        baseline_r2,
        linear_r2,
        gb_r2,
        rf_r2,
    ],
})

model_comparison

,Model,MAE_minutes,RMSE_minutes,R2
0,Baseline,5.211661,8.512903,-0.000015
1,Linear Regression,4.139913,7.525168,0.218582
2,Gradient Boosting,3.543673,7.324109,0.259780
3,Random Forest,3.777729,7.240638,0.276556


In [9]:
# ============================================================
# Cell 9 — Random Forest Error Analysis
# ============================================================

error_analysis = pd.DataFrame({
    "actual_minutes": y_test.values,
    "predicted_minutes": rf_predictions,
})

error_analysis["error_minutes"] = (
    error_analysis["predicted_minutes"]
    - error_analysis["actual_minutes"]
)

error_analysis["absolute_error_minutes"] = (
    error_analysis["error_minutes"].abs()
)

print("=== ERROR ANALYSIS ===")

print("\nMean Absolute Error:")
print(
    error_analysis["absolute_error_minutes"].mean()
)

print("\nLargest absolute errors:")
print(
    error_analysis
    .sort_values("absolute_error_minutes", ascending=False)
    .head(10)
)

print("\nError statistics:")
print(
    error_analysis["error_minutes"].describe()
)

=== ERROR ANALYSIS ===

Mean Absolute Error:
3.777728590487473

Largest absolute errors:
      actual_minutes  predicted_minutes  error_minutes  absolute_error_minutes
1424          130.00           6.553775    -123.446225              123.446225
3310          130.75          11.545031    -119.204969              119.204969
616           125.00          14.405953    -110.594047              110.594047
3355          131.50          26.052979    -105.447021              105.447021
5358          111.50          10.132244    -101.367756              101.367756
4103          109.75          14.266618     -95.483382               95.483382
262           103.25           8.859920     -94.390080               94.390080
2429          109.50          17.788287     -91.711713               91.711713
902           103.00          17.630328     -85.369672               85.369672
2195           94.25          24.839029     -69.410971               69.410971

Error statistics:
count    5728.000000
me

In [10]:
# ============================================================
# Cell 10 — Investigate Worst ETA Predictions
# ============================================================

worst_predictions = (
    df.loc[
        X_test.index,
        [
            "TRIP_ID",
            "TIMESTAMP",
            "trip_duration_minutes",
            "straight_line_distance_km",
            "start_longitude",
            "start_latitude",
            "end_longitude",
            "end_latitude",
            "hour",
            "day_of_week",
            "is_weekend",
            "CALL_TYPE",
            "ORIGIN_STAND",
        ],
    ]
    .copy()
)

worst_predictions["predicted_minutes"] = rf_predictions

worst_predictions["error_minutes"] = (
    worst_predictions["predicted_minutes"]
    - worst_predictions["trip_duration_minutes"]
)

worst_predictions["absolute_error_minutes"] = (
    worst_predictions["error_minutes"].abs()
)

worst_predictions = worst_predictions.sort_values(
    "absolute_error_minutes",
    ascending=False,
)

worst_predictions.head(10)

,TRIP_ID,TIMESTAMP,trip_duration_minutes,straight_line_distance_km,start_longitude,start_latitude,end_longitude,end_latitude,hour,day_of_week,is_weekend,CALL_TYPE,ORIGIN_STAND,predicted_minutes,error_minutes,absolute_error_minutes
24332,1399758237620000009,1399758237,130.00,0.682172,-8.628381,41.157171,-8.625816,41.151348,21,5,1,B,13.0,6.553775,-123.446225,123.446225
26218,1401745355620000009,1401745355,130.75,3.766804,-8.654967,41.177349,-8.625924,41.151474,21,0,0,C,NaN,11.545031,-119.204969,119.204969
23524,1399110872620000116,1399110872,125.00,2.507736,-8.610363,41.149269,-8.605755,41.126985,9,5,1,C,NaN,14.405953,-110.594047,110.594047
26263,1401797358620000450,1401797358,131.50,7.820079,-8.531415,41.148288,-8.612334,41.183433,12,1,0,C,NaN,26.052979,-105.447021,105.447021
28266,1403780408620000156,1403780408,111.50,2.601771,-8.602794,41.179626,-8.584785,41.160555,11,3,0,B,49.0,10.132244,-101.367756,101.367756
27011,1402503606620000504,1402503606,109.75,3.063240,-8.627940,41.157846,-8.613765,41.183244,16,2,0,B,13.0,14.266618,-95.483382,95.483382
23170,1398729716620000548,1398729716,103.25,0.358844,-8.609580,41.140692,-8.613792,41.141286,0,1,0,A,NaN,8.859920,-94.390080,94.390080
25337,1400846018620000527,1400846018,109.50,5.944981,-8.608797,41.147874,-8.582670,41.197590,11,4,0,B,27.0,17.788287,-91.711713,91.711713
23810,1399362775620000166,1399362775,103.00,3.879236,-8.593092,41.158917,-8.605269,41.125257,7,1,0,C,NaN,17.630328,-85.369672,85.369672
25103,1400575491620000408,1400575491,94.25,0.158135,-8.626068,41.155587,-8.626095,41.157009,8,1,0,C,NaN,24.839029,-69.410971,69.410971


In [11]:
# ============================================================
# Cell 11 — ETA Error Distribution
# ============================================================

abs_errors = error_analysis["absolute_error_minutes"]

error_summary = {
    "Within 2 min": (abs_errors <= 2).mean() * 100,
    "Within 5 min": (abs_errors <= 5).mean() * 100,
    "Within 10 min": (abs_errors <= 10).mean() * 100,
    "Over 10 min": (abs_errors > 10).mean() * 100,
    "Over 30 min": (abs_errors > 30).mean() * 100,
    "Over 60 min": (abs_errors > 60).mean() * 100,
}

print("=== ETA ERROR DISTRIBUTION ===")

for label, percentage in error_summary.items():
    print(f"{label:<15}: {percentage:.2f}%")

=== ETA ERROR DISTRIBUTION ===
Within 2 min   : 41.76%
Within 5 min   : 80.34%
Within 10 min  : 94.05%
Over 10 min    : 5.95%
Over 30 min    : 0.70%
Over 60 min    : 0.24%


In [12]:
# ============================================================
# Cell 12 — Save Production ETA Model
# ============================================================

import os
import joblib
import json

MODEL_DIR = "../../artifacts/eta_model"

os.makedirs(MODEL_DIR, exist_ok=True)

# Save the complete pipeline
model_path = os.path.join(
    MODEL_DIR,
    "eta_model.joblib"
)

joblib.dump(
    gradient_boosting_model,
    model_path
)

# Save model metadata
metadata = {
    "model_type": "GradientBoostingRegressor",
    "target": "trip_duration_minutes",
    "mae_minutes": float(gb_mae),
    "rmse_minutes": float(gb_rmse),
    "r2": float(gb_r2),
    "training_rows": int(len(X_train)),
    "testing_rows": int(len(X_test)),
    "features": FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "dataset": "UCI Porto Taxi Trajectory Dataset",
}

metadata_path = os.path.join(
    MODEL_DIR,
    "model_metadata.json"
)

with open(metadata_path, "w") as f:
    json.dump(
        metadata,
        f,
        indent=4
    )

print("=== MODEL ARTIFACT SAVED ===")
print(f"Model   : {os.path.abspath(model_path)}")
print(f"Metadata: {os.path.abspath(metadata_path)}")

=== MODEL ARTIFACT SAVED ===
Model   : C:\Users\devra\OneDrive\Desktop\Devraj_coding\MissionFlow-AI\artifacts\eta_model\eta_model.joblib
Metadata: C:\Users\devra\OneDrive\Desktop\Devraj_coding\MissionFlow-AI\artifacts\eta_model\model_metadata.json


In [13]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nModel exists:")
print(os.path.exists("../../artifacts/eta_model/eta_model.joblib"))

print("\nSaved model path:")
print(os.path.abspath("../../artifacts/eta_model/eta_model.joblib"))

Current working directory:
C:\Users\devra\OneDrive\Desktop\Devraj_coding\MissionFlow-AI\ml\notebooks

Model exists:
True

Saved model path:
C:\Users\devra\OneDrive\Desktop\Devraj_coding\MissionFlow-AI\artifacts\eta_model\eta_model.joblib


In [14]:
# ============================================================
# Cell 14 — Move ETA Model Artifact to Correct Location
# ============================================================

import os
import shutil

SOURCE_DIR = "../../artifacts/eta_model"
TARGET_DIR = "../artifacts/eta_model"

os.makedirs(TARGET_DIR, exist_ok=True)

for filename in [
    "eta_model.joblib",
    "model_metadata.json",
]:
    source = os.path.join(SOURCE_DIR, filename)
    target = os.path.join(TARGET_DIR, filename)

    shutil.copy2(source, target)

print("=== ETA MODEL ARTIFACT MOVED ===")
print("Target directory:")
print(os.path.abspath(TARGET_DIR))

print("\nFiles:")
for filename in os.listdir(TARGET_DIR):
    print(f" - {filename}")

=== ETA MODEL ARTIFACT MOVED ===
Target directory:
C:\Users\devra\OneDrive\Desktop\Devraj_coding\MissionFlow-AI\ml\artifacts\eta_model

Files:
 - eta_model.joblib
 - model_metadata.json


In [15]:
# ============================================================
# Cell 15 — Reload Saved ETA Model
# ============================================================

import os
import joblib
import json

MODEL_PATH = "../artifacts/eta_model/eta_model.joblib"
METADATA_PATH = "../artifacts/eta_model/model_metadata.json"

# Load model
loaded_eta_model = joblib.load(MODEL_PATH)

# Load metadata
with open(METADATA_PATH, "r") as f:
    loaded_metadata = json.load(f)

print("=== ETA MODEL RELOADED ===")
print(f"Model type : {loaded_metadata['model_type']}")
print(f"Target     : {loaded_metadata['target']}")

print("\n=== SAVED PERFORMANCE ===")
print(f"MAE  : {loaded_metadata['mae_minutes']:.3f} minutes")
print(f"RMSE : {loaded_metadata['rmse_minutes']:.3f} minutes")
print(f"R²   : {loaded_metadata['r2']:.3f}")

print("\n=== ARTIFACT STATUS ===")
print(f"Model exists    : {os.path.exists(MODEL_PATH)}")
print(f"Metadata exists : {os.path.exists(METADATA_PATH)}")

=== ETA MODEL RELOADED ===
Model type : GradientBoostingRegressor
Target     : trip_duration_minutes

=== SAVED PERFORMANCE ===
MAE  : 3.544 minutes
RMSE : 7.324 minutes
R²   : 0.260

=== ARTIFACT STATUS ===
Model exists    : True
Metadata exists : True
